In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
import pandas as pd
from common.const import SEMANTIC_VECTOR_DB_COLLECTION, SEMANTIC_VECTOR_DB_COLLECTION_USER
from common.const import DEVICE, SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS, QDRANT_CLUSTER_ENDPOINT_MINH_HOANG
from common.helper import clickhouse_extract_data_as_df, connect_qdrant, connect_clickhouse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
from qdrant_client.models import PayloadSchemaType
qd_client = connect_qdrant(
    url=QDRANT_CLUSTER_ENDPOINT_MINH_HOANG,
    api_key=os.getenv("QDRANT_API_KEY")
)

c:\Users\Admin\.conda\envs\scc454\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


# 1. Query data

In [2]:
# Products
products_query = f"""
SELECT
    parent_asin,
    title,
    description,
    features,
    main_category,
    CAST(price AS Nullable(Float64)) AS price,
    average_rating,
    rating_number
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
LIMIT 100000
"""
products_df = clickhouse_extract_data_as_df(products_query)

# Feature Engineering
def create_product_text(row):
    text_parts = []
    
    text_parts.append(row['title'] + (" " + row['title']) * 1)
    text_parts.append(row['description'])
    text_parts.append(row['features'])
    text_parts.append(row['main_category'] + (" " + row['main_category']) * 2)
    
    return ' '.join(text_parts)

products_df['combined_text'] = products_df.apply(create_product_text, axis=1)

# Reviews
reviews_query = f"""
SELECT 
    user_id,
    parent_asin,
    title,
    text,
    rating
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
LIMIT 100000
"""
reviews_df = clickhouse_extract_data_as_df(reviews_query)

In [3]:
products_df

,parent_asin,title,description,features,main_category,price,average_rating,rating_number,combined_text
0,0005946468,Patanjali Moisturizer Cream - Pack of 2,,,All Beauty,NaN,5.0,2,Patanjali Moisturizer Cream - Pack of 2 Patanj...
1,0123034892,Yardley Lace Satin Deodorant Spray for women 1...,,,All Beauty,NaN,5.0,1,Yardley Lace Satin Deodorant Spray for women 1...
2,0124784577,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,,,All Beauty,NaN,3.8,15,WOW Organics Apple Cider Vinegar Shampoo - 300...
3,0515059560,Love on the Run,,,All Beauty,NaN,4.0,1,Love on the Run Love on the Run All Beauty A...
4,0692508988,The Listening Cards,The Listening Cards are an eloquent primer in ...,,All Beauty,NaN,5.0,1,The Listening Cards The Listening Cards The Li...
...,...,...,...,...,...,...,...,...,...
99995,B0063HP6C6,MAHLER: SYM NO.7(2SACD hybrid),SUPER AUDIO CD,,Digital Music,56.51,4.5,3,MAHLER: SYM NO.7(2SACD hybrid) MAHLER: SYM NO....
99996,B0063HPDL0,Hensley-Ayres,Remember The Times Better Part Of Me Can It Su...,,Digital Music,NaN,5.0,2,Hensley-Ayres Hensley-Ayres Remember The Times...
99997,B0063I5L20,KFOG Live From the Archives 18,"1. Scars On 45 - ""Give Me Something"" Live @ Ex...",,Digital Music,5.00,4.7,4,KFOG Live From the Archives 18 KFOG Live From ...
99998,B0063IZMHO,Stare polskie koledy - Old Polish Christmas Ca...,,,Digital Music,NaN,4.5,2,Stare polskie koledy - Old Polish Christmas Ca...


# 2. Generate similarity computation for similarity product

## 2.1. SentenceTransformer

In [4]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings (384 dimensions)
print("Encoding products (this may take a few minutes)...")
semantic_embeddings = model.encode(
    products_df['combined_text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,  # Normalize for cosine similarity
    device=DEVICE
)
print(f"Semantic embeddings shape: {semantic_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding products (this may take a few minutes)...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Semantic embeddings shape: (100000, 384)


## 2.2. Insert vectors

In [5]:
# Delete the collection
qd_client.delete_collection(collection_name=SEMANTIC_VECTOR_DB_COLLECTION)

# Create collection
qd_client.create_collection(
    collection_name=SEMANTIC_VECTOR_DB_COLLECTION,
    vectors_config=VectorParams(
        size=384,  # MiniLM embedding size
        distance=Distance.COSINE
    ),
    quantization_config={
        "scalar": {
            "type": "int8",
            "quantile": 0.99,
            "always_ram": True
        }
    }
)
# Add points
points = [
    PointStruct(
        id=idx,
        vector=semantic_embeddings[idx].tolist(),
        payload={
            "product_asin": row['parent_asin'],
            "title": row['title'],
            "main_category": row['main_category'],
            "price": row['price']
        }
    )
    for idx, row in products_df.iterrows()
]

BATCH_SIZE = 500 # Reduced batch size for better memory management and avoid Qdrant server closed connection
for i in range(0, len(points), BATCH_SIZE):
    qd_client.upsert(
        collection_name=SEMANTIC_VECTOR_DB_COLLECTION,
        points=points[i:i + BATCH_SIZE]
    )

## 2.3. Query similar products

In [6]:
def get_similar_products(product_asin, k=5, filters=None):
    """
    Find k most similar products
    
    Args:
        product_asin: Product ID to find similar items for
        k: Number of similar products to return
        filters: Optional Qdrant filter conditions
    
    Returns:
        List of similar products with scores
    """
    # Find the product
    product_row = products_df[products_df['parent_asin'] == product_asin]
    
    if product_row.empty:
        return f"Product {product_asin} not found"
    
    product_idx = product_row.index[0]
    query_vector = semantic_embeddings[product_idx].tolist()
    
    # Search in Qdrant
    search_result = qd_client.query_points(
        collection_name=SEMANTIC_VECTOR_DB_COLLECTION,
        query=query_vector,
        limit=k+1,
        with_payload=True
    ).points
    
    # Format results (skip first one - the query product itself)
    results = []
    for hit in search_result[1:]:
        results.append({
            'product_asin': hit.payload['product_asin'],
            'title': hit.payload['title'],
            'similarity_score': hit.score,
            'price': hit.payload.get('price'),
            'rating': hit.payload.get('average_rating'),
            'category': hit.payload['main_category']
        })
    
    return results


sample_product = "0001048252"
get_similar_products(sample_product)

[{'product_asin': 'B002Q1Y96M',
  'title': 'The Original Beverly Hillbillies: The Original',
  'similarity_score': 0.45114982,
  'price': None,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B002Q2XZ5C',
  'title': 'Goodbye Pinto',
  'similarity_score': 0.42841488,
  'price': None,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B0028S2V90',
  'title': 'Images',
  'similarity_score': 0.40892777,
  'price': 39.95,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B002Q7ZP4G',
  'title': 'Foy Willing: The Riders Of The Purple Sage',
  'similarity_score': 0.39877874,
  'price': None,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B005BJPDPW',
  'title': 'Not For Horses',
  'similarity_score': 0.3977598,
  'price': 64.82,
  'rating': None,
  'category': 'Digital Music'}]

In [7]:
# Example 2: Filtered similarity search (Qdrant's killer feature!)
# Find similar products within a price range
price_filter = Filter(
    must=[
        FieldCondition(
            key="price",
            range=Range(gte=10.0, lte=100.0)  # Products between $10-$100
        )
    ]
)

similar_affordable = get_similar_products(sample_product, k=10, filters=price_filter)
print(f"\n=== Similar affordable products ($10-$100) ===")
for i, product in enumerate(similar_affordable, 1):
    print(f"{i}. {product['product_asin']} - ${product['price']}")


=== Similar affordable products ($10-$100) ===
1. B002Q1Y96M - $None
2. B002Q2XZ5C - $None
3. B0028S2V90 - $39.95
4. B002Q7ZP4G - $None
5. B005BJPDPW - $64.82
6. B005CYURYI - $None
7. B002QO4JNC - $None
8. B00LAC4RLU - $None
9. B000HFOW8E - $29.99
10. B003HEBGMM - $None


In [8]:
# Example 3: Search by vector directly (useful for cold-start)
def search_by_query_text(query_text, k=10):
    """Search products by text query"""
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    query_vector = model.encode([query_text], normalize_embeddings=True, device=DEVICE)
    query_vector = query_vector[0].tolist()
    
    # query_vector = vectorizer.transform().toarray().astype('float32')
    # query_vector = normalize(query_vector, norm='l2')[0].tolist()
    
    # Search in Qdrant
    search_result = qd_client.query_points(
        collection_name=SEMANTIC_VECTOR_DB_COLLECTION,
        query=query_vector,
        limit=k+1,
        with_payload=True
    ).points
    
    results = []
    for hit in search_result:
        results.append({
            'product_asin': hit.payload['product_asin'],
            'title': hit.payload['title'],
            'similarity_score': hit.score,
            'price': hit.payload.get('price'),
            'rating': hit.payload.get('average_rating'),
            'category': hit.payload['main_category']
        })
    
    return results

search_by_query_text("wireless headphones bluetooth", k=5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'product_asin': 'B000BAVPH6',
  'title': 'Bluetooth 5.87 Earbuds Deep Bass in-Ear Bluetooth Headset Hi-Fi Stereo',
  'similarity_score': 0.5706444,
  'price': 12.15,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B014VTGC9I',
  'title': 'Wireless Earbuds, 455D Stereo Sound Wireless Headphones Wireless Sport Earbud with Breathing Mini in-Ear Sports Earphones Noise Cancelling Headsets, Bluetooth Earbuds',
  'similarity_score': 0.56243706,
  'price': None,
  'rating': None,
  'category': 'All Beauty'},
 {'product_asin': 'B000GS5TFM',
  'title': 'Bluetooth Headphones, Ul-12 Lightweight & Fast Sports Earphones IPX7 Waterproof HD Stereo Sweatproof Earbuds Noise Cancelling Headsets Gym Running (UL12)',
  'similarity_score': 0.47408676,
  'price': None,
  'rating': None,
  'category': 'Digital Music'},
 {'product_asin': 'B073W316HX',
  'title': 'Headphone Adapter',
  'similarity_score': 0.45596504,
  'price': None,
  'rating': None,
  'category': 'All Beauty'},
 {'produ

# 3. Generate similarity computation for similar users

## 3.1. Feature engineering

In [9]:
#for similar user, we want user with same behaviour, not similar writing style. And rn, we want it to be separated with item embeddings
#so I'll use collaborative filtering
#from that, build user-item interaction matrix -> factorize it -> get user latent vector 
# -> Compute rating statistics per user (e.g. average rating, rating count)
# -> compute avg text embedding per user
# -> combine vector and store it in Qdrant

In [10]:
reviews_df

,user_id,parent_asin,title,text,rating
0,AG25WBN7G5F6NMVC4AX7NDSA2LLA,B07RLNV8RL,Works as described,Works as described. I can’t speak for the long...,4
1,AGO3JLXSVYTR66HXSYCQXSZNQRPA,B07RLNV8RL,It’s quiet and it shaves - but not very well.,You get what you pay for. Cheap price = low p...,1
2,AEHRUSNF4JG5222J44VJC5JTDY4A,B07RLNV8RL,"Closer, faster shave",Exceeded expectations. Granted new blades hel...,5
3,AHAEVHNHX44R3UAXVBS3CIQCPN6Q,B07RLNV8RL,Buy with confidence,This brand is quality. Excellent product,5
4,AFM5EPS643NJOBLI646XFYQ2TSIQ,B07RLNV8RL,You won't regret it!!,Amazing product! I didn't think razors could w...,5
...,...,...,...,...,...
99995,AHGAVL2RWFLW4WQT7FCEB2AXHPRA,B01DD1NOZU,Great for my locs.,I purchased the almond milk leave in conditio...,5
99996,AFCCPOFPEZ4EK534UG3QCXY7X7LQ,B01DD1NOZU,It’s moisturizing!,The best leave in conditioner.,5
99997,AFVRDSBJM4ZN2VF33FIW5M5YIX5Q,B01DD1NOZU,Love it,Love all of Carol's Daughter's products,5
99998,AFJWAP5LJKNGL563A4KKZM6DYLVQ,B01DD1NOZU,My hair was easier to manage after I used this...,I loved using Carol's Daughter almond milk lea...,4


In [11]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares

# --- 1. Encode users/items ---
reviews_df['user_idx'] = reviews_df['user_id'].astype("category").cat.codes
reviews_df['item_idx'] = reviews_df['parent_asin'].astype("category").cat.codes

num_users = reviews_df['user_idx'].nunique()
num_items = reviews_df['item_idx'].nunique()

# --- 2. Build sparse user-item matrix ---
# Keep all ratings (1 to 5) as explicit feedback
rows = reviews_df['user_idx'].values
cols = reviews_df['item_idx'].values
ratings = reviews_df['rating'].astype(float).values

user_item_matrix = coo_matrix(
    (ratings, (rows, cols)),
    shape=(num_users, num_items)
) #tbf, idk why we do this then transpose then change back, but other reference do it.

# ALS in implicit library expects item-user matrix
item_user_matrix = user_item_matrix.T.tocsr()

# --- 3. Train ALS in explicit mode ---
als_model = AlternatingLeastSquares(
    factors=384,          # number of latent factors, same as embedding size for easy combination later
    regularization=0.1,
    iterations=20,
    calculate_training_loss=True,
    use_gpu=True if DEVICE == 'cuda' else False
)

als_model.fit(item_user_matrix)  # item-user matrix

# --- 4. Extract user/item latent vectors ---
# I input item_user_matrix Item x User so:
# als_model.user_factors is ITEM vectors
# als_model.item_factors is USER vectors

item_vectors_data = als_model.user_factors  #(14663, 384)
user_vectors_data = als_model.item_factors  #(96746, 384)

# --- 5. Map back ID ---
user_codes = reviews_df['user_id'].astype("category").cat.categories
item_codes = reviews_df['parent_asin'].astype("category").cat.categories

user_latent_vectors = pd.DataFrame(user_vectors_data)
user_latent_vectors['user_id'] = user_codes

item_latent_vectors = pd.DataFrame(item_vectors_data)
item_latent_vectors['parent_asin'] = item_codes

print(f"User vectors shape: {user_latent_vectors.shape}") #should be (96746, 385)
print(f"Item vectors shape: {item_latent_vectors.shape}") #should be (14663, 385) (384 + 1 ID)


c:\Users\Admin\.conda\envs\scc454\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

User vectors shape: (96746, 385)
Item vectors shape: (14663, 385)


In [12]:
# Combine title + text first
reviews_df["combined_text"] = (
    reviews_df["title"].fillna("") + " " +
    reviews_df["text"].fillna("")
)

# Aggregate by user
user_text_df = (
    reviews_df
    .groupby("user_id")["combined_text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

user_text_df

,user_id,combined_text
0,AE223QFU63B3X4P437QAGGX5V3CQ,"Five Stars Smells awesome, works great. Keeps ..."
1,AE225P7FIMCUQD3TPAOF5LSGCLYQ,Tastes good Good strawberry flavor that isn't ...
2,AE225THXPLSO5QQ3PHQP7SPETM2A,Very beautiful shade! I loved this shade. It w...
3,AE22C7UR3G223VSKR3EDUTQZD6ZQ,Love it Tiny but great for travelling pouch an...
4,AE22DEFZFLUMM6QYGOS77AZU7NKA,"Yes Airy, light, silky."
...,...,...
96741,AHZZXE5OWOEY2WDWGIF26YMI6ORQ,"My new favorite So beautiful. I love this cd,..."
96742,AHZZXJTF7F6A3NY2BV65DZ4ZHBRA,Disappointed: Doesn't fit Ordered these for my...
96743,AHZZXYROONDALZSG7DO6VAT3VUZQ,"Asking $1000 for a $10 CD, just listen to spot..."
96744,AHZZZ6OLGGSAUFFBL2W5FRPVVPPQ,Not durable! I bought this in July. After 2 m...


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Encoding user text...")

text_embeddings = model.encode(
    user_text_df["combined_text"].tolist(),
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,  # important
    device=DEVICE
)

print("Text embedding shape:", text_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding user text...


Batches:   0%|          | 0/3024 [00:00<?, ?it/s]

Text embedding shape: (96746, 384)


In [14]:
rating_stats = (
    reviews_df
    .groupby("user_id")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count"),
        rating_std=("rating", "std"),
        five_star_ratio=("rating", lambda x: (x == 5).mean())
    )
    .fillna(0)
    .reset_index()
)

rating_stats

,user_id,avg_rating,rating_count,rating_std,five_star_ratio
0,AE223QFU63B3X4P437QAGGX5V3CQ,5.0,1,0.000000,1.0
1,AE225P7FIMCUQD3TPAOF5LSGCLYQ,4.0,1,0.000000,0.0
2,AE225THXPLSO5QQ3PHQP7SPETM2A,3.5,2,0.707107,0.0
3,AE22C7UR3G223VSKR3EDUTQZD6ZQ,5.0,1,0.000000,1.0
4,AE22DEFZFLUMM6QYGOS77AZU7NKA,5.0,1,0.000000,1.0
...,...,...,...,...,...
96741,AHZZXE5OWOEY2WDWGIF26YMI6ORQ,5.0,1,0.000000,1.0
96742,AHZZXJTF7F6A3NY2BV65DZ4ZHBRA,1.0,1,0.000000,0.0
96743,AHZZXYROONDALZSG7DO6VAT3VUZQ,1.0,1,0.000000,0.0
96744,AHZZZ6OLGGSAUFFBL2W5FRPVVPPQ,2.0,1,0.000000,0.0


In [15]:
from sklearn.preprocessing import normalize, StandardScaler

# Lấy danh sách user_id đúng thứ tự từ ALS
ordered_user_ids = user_latent_vectors['user_id'].tolist()

# 1. Map Text Embeddings
# Chuyển text_embeddings (numpy array) thành DF để dễ map
text_emb_df = pd.DataFrame(text_embeddings, index=user_text_df['user_id'])
# Reindex theo thứ tự ALS, những user thiếu text sẽ bị NaN (ta fill 0)
text_emb_ordered = text_emb_df.reindex(ordered_user_ids).fillna(0).values

# 2. Map Rating Stats
# Làm tương tự với rating_stats
rating_stats_indexed = rating_stats.set_index('user_id')
rating_stats_ordered_df = rating_stats_indexed.reindex(ordered_user_ids).fillna(0)

# Scale rating stats (chỉ lấy các cột số)
scaler = StandardScaler()
rating_values_scaled = scaler.fit_transform(rating_stats_ordered_df)

In [16]:
# Lấy phần số của ALS (bỏ cột user_id)
als_vectors_np = user_latent_vectors.drop(columns=['user_id']).values

# Normalize từng phần
als_norm = normalize(als_vectors_np)
text_norm = normalize(text_emb_ordered)

# Kết hợp (Weighted Sum)
combined_vectors = (0.8 * als_norm) + (0.2 * text_norm)

# Final normalize cho Cosine Distance
final_vectors = normalize(combined_vectors)

## 3.2. Insert Vectors

In [17]:
# Delete the collection
qd_client.delete_collection(collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER)

# Create collection
qd_client.create_collection(
    collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER,
    vectors_config=VectorParams(
        size=384,  # MiniLM embedding size
        distance=Distance.COSINE
    ),
    quantization_config={
        "scalar": {
            "type": "int8",
            "quantile": 0.99,
            "always_ram": True
        }
    }
)

final_df = rating_stats_ordered_df.reset_index()
final_df['vector'] = final_vectors.tolist()
# Add points
points = [
    PointStruct(
        id=idx,
        vector=row['vector'],
        payload={
            "user_id": row['user_id'],
            "avg_rating": float(row['avg_rating']),
            "rating_count": int(row['rating_count']),
            "five_star_ratio": float(row['five_star_ratio'])
        }
    )
    for idx, row in final_df.iterrows()
]

BATCH_SIZE = 500 # Reduced batch size for better memory management and avoid Qdrant server closed connection
for i in range(0, len(points), BATCH_SIZE):
    qd_client.upsert(
        collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER,
        points=points[i:i + BATCH_SIZE]
    )

## 3.3. Query similar User

In [18]:
def get_similar_users(user_id, k=5, filters=None):
    #copy from product, but change to user
    user_row = final_df[final_df['user_id'] == user_id]
    
    if user_row.empty:
        return f"User {user_id} not found in memory."
    
    query_vector = user_row['vector'].values[0]
    
    try:
        search_result = qd_client.query_points(
            collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER,
            query=query_vector,
            limit=k + 1,
            query_filter=filters,
            with_payload=True
        ).points
    except Exception as e:
        return f"Error querying Qdrant: {str(e)}"
    
    results = []
    for hit in search_result:
        if hit.payload['user_id'] == user_id:
            continue
            
        results.append({
            'user_id': hit.payload['user_id'],
            'similarity_score': round(hit.score, 4),
            'avg_rating': hit.payload.get('avg_rating'),
            'rating_count': hit.payload.get('rating_count'),
            'five_star_ratio': hit.payload.get('five_star_ratio')
        })
    
    # limit k
    return results[:k]

sample_user = "AGVLZZGOXKCJKSQDG5T6EZZWMRMA"
similar_users = get_similar_users(sample_user, k=5)

if isinstance(similar_users, list):
    print(f"user similar with {sample_user}:")
    display(pd.DataFrame(similar_users))
else:
    print(similar_users)


user similar with AGVLZZGOXKCJKSQDG5T6EZZWMRMA:


,user_id,similarity_score,avg_rating,rating_count,five_star_ratio
0,AENVFXKDCWH5NRNEKXKHAGDSVP7A,0.9975,5.0,1,1.0
1,AHDKG4U2KADYDADDXUXC4Y6RAAPA,0.9969,5.0,1,1.0
2,AGLM3SJHQU4K2ACHVG6OWCKPHGYQ,0.9951,5.0,1,1.0
3,AGTZY2QWEQTJE3E66U4XWMWDUIOA,0.9942,5.0,1,1.0
4,AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ,0.9940,5.0,1,1.0


In [19]:
def get_similar_user_details(sample_user_id, k=5):
    similar_users_res = get_similar_users(sample_user_id, k=k)
    if isinstance(similar_users_res, list):
        #get list ID
        similar_ids = [u['user_id'] for u in similar_users_res]
        
        #create clickhouse query
        ids_str = ", ".join([f"'{i}'" for i in similar_ids])
        
        final_query = f"""
        SELECT 
            user_id,
            parent_asin,
            title,
            text,
            rating
        FROM {SCHEMA}.{TABLE_RAW_REVIEWS} 
        WHERE user_id IN ({ids_str})
        ORDER BY indexOf([{ids_str}], user_id) ASC, rating DESC
        """
        
        print("--- ClickHouse Query ---")
        # print(final_query)
        client = connect_clickhouse()
        df = client.query_df(final_query)
        client.close()
    else:
        print("similar user not found")
    return df

get_similar_user_details(sample_user, k=5)

--- ClickHouse Query ---


,user_id,parent_asin,title,text,rating
0,AENVFXKDCWH5NRNEKXKHAGDSVP7A,B01BN1U8FQ,Five Stars,Smells amazing!,5
1,AHDKG4U2KADYDADDXUXC4Y6RAAPA,B01BISG6SC,Five Stars,great smell,5
2,AGLM3SJHQU4K2ACHVG6OWCKPHGYQ,B01AVERN9K,Five Stars,Smells great!,5
3,AGTZY2QWEQTJE3E66U4XWMWDUIOA,B01CXA2VFG,Five Stars,smell good,5
4,AGTZY2QWEQTJE3E66U4XWMWDUIOA,B01KGCOQFQ,Effective for people with ezcema.,I bought this for my daughter it helped with h...,5
5,AGTZY2QWEQTJE3E66U4XWMWDUIOA,B0055I68IA,Four Stars,g00d,4
6,AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ,B01BK49MZI,Five Stars,Smells great. Good quality.,5
7,AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ,B01LYRATO0,This is the best face wash I've tried,This is the best face wash I've tried. It leav...,5
8,AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ,B07JNG7YZW,Great for finals!,I bought these for my college student to help ...,5
9,AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ,B0765BXL28,This is my new favorite deodorant. It leaves a...,This is my new favorite deodorant. It leaves a...,5


In [20]:
#create payload index cuz data too big otherwise 400 error
qd_client.create_payload_index(
    collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER,
    field_name="avg_rating",
    field_schema=PayloadSchemaType.FLOAT,
)
#not tested yet, but for other field, you have to create index as well

user_filter = Filter(
    must=[
        FieldCondition(
            key="avg_rating",
            range=Range(gte=3.0)  #example filter
        )
    ]
)

sample_user_id = sample_user

similar_power_users = get_similar_users(
    sample_user_id, 
    k=10, 
    filters=user_filter
)
similar_power_users
print(f"=== Similar Power Users (Rating Count >= 5 & Avg Rating >= 3.0) ===")
for i, user in enumerate(similar_power_users, 1):
    print(f"{i}. {user['user_id']} | Score: {user['similarity_score']:.4f} | Reviews: {user['rating_count']} | Avg: {user['avg_rating']}")

=== Similar Power Users (Rating Count >= 5 & Avg Rating >= 3.0) ===
1. AENVFXKDCWH5NRNEKXKHAGDSVP7A | Score: 0.9980 | Reviews: 1 | Avg: 5.0
2. AHDKG4U2KADYDADDXUXC4Y6RAAPA | Score: 0.9969 | Reviews: 1 | Avg: 5.0
3. AGLM3SJHQU4K2ACHVG6OWCKPHGYQ | Score: 0.9951 | Reviews: 1 | Avg: 5.0
4. AGSZYF7JUB4VMQRCHVQ6JTIJOVTQ | Score: 0.9948 | Reviews: 1 | Avg: 5.0
5. AGTZY2QWEQTJE3E66U4XWMWDUIOA | Score: 0.9936 | Reviews: 1 | Avg: 5.0
6. AGO4A5XEPGUYIW7BMICFWRD2WUHA | Score: 0.9930 | Reviews: 1 | Avg: 5.0
7. AHSOOGKFP3HBWLSZZBGU5LP3J4IA | Score: 0.9927 | Reviews: 1 | Avg: 5.0
8. AEGZRSDMEQMKAHVZ5IJBQKSKTZPA | Score: 0.9926 | Reviews: 1 | Avg: 5.0
9. AGROBKV6H2UQQHZDTLGL6HVCPH7Q | Score: 0.9921 | Reviews: 1 | Avg: 5.0
10. AEJ42HWW6S63BINVBYVAZ54UHYIQ | Score: 0.9919 | Reviews: 1 | Avg: 5.0


In [21]:
def search_users_by_interest(query_text, k=5):
    """input text, output similar user id, sim score, avg_rating"""
    
    #Encode query text (384 dim)
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    text_v = model.encode([query_text], normalize_embeddings=True, device=DEVICE)[0]
    query_vector = text_v.tolist()
    
    #Search collection USER
    search_result = qd_client.query_points(
        collection_name=SEMANTIC_VECTOR_DB_COLLECTION_USER,
        query=query_vector,
        limit=k,
        with_payload=True
    ).points
    
    #format
    results = []
    for hit in search_result:
        results.append({
            'user_id': hit.payload['user_id'],
            'similarity_score': hit.score,
            'avg_rating': hit.payload.get('avg_rating'),
        })
    return results

def get_user_interest_details(query_text, k=5):
    # input the list
    similar_users = search_users_by_interest(query_text, k=k)
    
    if not similar_users:
        return "not found similar user"

    # get user id list for query
    ids = [u['user_id'] for u in similar_users]
    ids_str = ", ".join([f"'{uid}'" for uid in ids])

    #ClickHouse order by indexof
    query = f"""
    SELECT 
        user_id,
        parent_asin,
        title,
        rating,
        text,
        timestamp
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE user_id IN ({ids_str})
    ORDER BY indexOf([{ids_str}], user_id) ASC, rating DESC
    """

    client = connect_clickhouse()
    df_details = client.query_df(query)
    client.close()
    # 4. Map similarity_score 2 dataframe
    scores_map = {u['user_id']: u['similarity_score'] for u in similar_users}
    df_details['similarity_score'] = df_details['user_id'].map(scores_map)
    
    return df_details


result_df = get_user_interest_details("I love high-end electronics and detailed reviews", k=5)
display(result_df)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,user_id,parent_asin,title,rating,text,timestamp,similarity_score
0,AG7YZXCBWDZDG2X6V3ASR2U6ZQFQ,B01D9K4HME,Five Stars,5,Great quality and price,1523904921128,0.215614
1,AEYWD4Q5IVBZ7RZGI5QHYPYAOD3Q,B01D9K4HME,Cheap,3,cheaply made,1588492919083,0.212534
2,AGCHXD7UHXZGBUOZR4I54BROSA5A,B07TWKWGFV,I love Amazon prime,5,This is the best thing I bought so far and I l...,1581914851249,0.209315
3,AFGM37FZCYLBDADREMAX34JECJCQ,B07TWKWGFV,Does the job. Quality for good price.,5,Good product.,1581407053793,0.204817
4,AHDZHRZQWQLVSDKQXCPYFUMQY73Q,B07RX25PSS,Great quality!,5,Better than I imagined! Great dupe to the 180$...,1562995646503,0.204507
